# Sola Face LoRA (SDXL) — Colab

1. **Runtime → GPU** (перевір `!nvidia-smi`)
2. Upload `foocus_new/datasets/sola_face_kohya.zip`
3. Run cells **по черзі** (не Skip train)
4. Download `.safetensors`

Trigger: `sola_face`


In [ ]:
# @title 0) GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Увімкни Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0), "VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")


In [ ]:
# @title 1) Setup Kohya sd-scripts
import os
os.chdir("/content")
!pip -q install -U pip
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
if not os.path.isdir("/content/sd-scripts"):
    !git clone --depth 1 -b sd3 https://github.com/kohya-ss/sd-scripts /content/sd-scripts || git clone --depth 1 https://github.com/kohya-ss/sd-scripts /content/sd-scripts
os.chdir("/content/sd-scripts")
!pip -q install -r requirements.txt
!pip -q install bitsandbytes==0.43.3 accelerate==0.33.0 transformers==4.44.2 diffusers==0.30.0 safetensors
!accelerate config default
print("OK setup", os.getcwd())


In [ ]:
# @title 2) Upload sola_face_kohya.zip + normalize
import os, zipfile, shutil
from google.colab import files

DATA = "/content/sola_data"
shutil.rmtree(DATA, ignore_errors=True)
os.makedirs(DATA, exist_ok=True)
os.chdir(DATA)

print("Upload: foocus_new/datasets/sola_face_kohya.zip")
uploaded = files.upload()
assert uploaded, "No file uploaded"

for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(os.path.join(DATA, name)) as z:
            z.extractall(DATA)
            print("zip entries", len(z.namelist())[:], "sample", z.namelist()[:5])

found = None
for root, dirs, files_ in os.walk(DATA):
    for d in dirs:
        if d == "10_sola_face" or (d.startswith("10_") and "sola" in d.lower()):
            found = os.path.join(root, d)
            break
    if found:
        break

if found is None:
    jpgs = []
    for root, dirs, files_ in os.walk(DATA):
        for f in files_:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                jpgs.append(os.path.join(root, f))
    assert len(jpgs) >= 10, f"Bad zip, images={len(jpgs)}. Upload sola_face_kohya.zip"
    found = os.path.join(DATA, "10_sola_face")
    os.makedirs(found, exist_ok=True)
    for src in jpgs:
        stem = os.path.splitext(os.path.basename(src))[0]
        shutil.copy2(src, os.path.join(found, stem + ".jpg"))
        txt_src = os.path.splitext(src)[0] + ".txt"
        txt_dst = os.path.join(found, stem + ".txt")
        if os.path.isfile(txt_src):
            shutil.copy2(txt_src, txt_dst)
        else:
            open(txt_dst, "w", encoding="utf-8").write(
                "sola_face, photo of a woman, adult woman, long straight brown hair, middle part, looking at camera\n"
            )

TRAIN_ROOT = os.path.dirname(found)
CLASS_DIR = found
OUT = "/content/outputs/sola_face_lora"
os.makedirs(OUT, exist_ok=True)
n_img = len([f for f in os.listdir(CLASS_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
print("TRAIN_ROOT", TRAIN_ROOT)
print("CLASS_DIR", CLASS_DIR, "images", n_img)
assert n_img >= 10


In [ ]:
# @title 3) Train SDXL LoRA (checks exit code)
import os, subprocess, glob

os.chdir("/content/sd-scripts")
LOG = "/content/outputs/sola_face_lora/train.log"
os.makedirs(OUT, exist_ok=True)

# 768 is safer on free T4; bump to 1024 on L4/A100 if you want
RES = "768,768"

cmd = [
    "accelerate", "launch", "--num_cpu_threads_per_process", "2",
    "sdxl_train_network.py",
    "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
    f"--train_data_dir={TRAIN_ROOT}",
    f"--output_dir={OUT}",
    "--output_name=sola_face_sdxl",
    "--save_model_as=safetensors",
    "--save_precision=fp16",
    "--caption_extension=.txt",
    f"--resolution={RES}",
    "--enable_bucket",
    "--min_bucket_reso=512",
    "--max_bucket_reso=1536",
    "--train_batch_size=1",
    "--gradient_checkpointing",
    "--max_train_epochs=8",
    "--save_every_n_epochs=2",
    "--learning_rate=1e-4",
    "--unet_lr=1e-4",
    "--text_encoder_lr=5e-5",
    "--lr_scheduler=cosine_with_restarts",
    "--lr_scheduler_num_cycles=2",
    "--optimizer_type=AdamW8bit",
    "--network_module=networks.lora",
    "--network_dim=16",
    "--network_alpha=8",
    "--mixed_precision=fp16",
    "--cache_latents",
    "--cache_latents_to_disk",
    "--seed=42",
    "--keep_tokens=1",
    "--noise_offset=0.0357",
    "--min_snr_gamma=5",
    "--max_data_loader_n_workers=0",
]

print("Running:", " ".join(cmd))
with open(LOG, "w", encoding="utf-8") as logf:
    proc = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, text=True)

print("exit_code =", proc.returncode)
print("--- tail of train.log ---")
!tail -n 80 {LOG}

paths = sorted(glob.glob(OUT + "/**/*.safetensors", recursive=True))
print("safetensors:", paths)
if proc.returncode != 0 or not paths:
    raise RuntimeError(
        "Training failed or produced no LoRA. Scroll train.log above. "
        "Common: OOM → Runtime high-RAM GPU; missing accelerate; dataset path."
    )


In [ ]:
# @title 4) Find + download ANY LoRA under /content
import glob, os
from google.colab import files

paths = sorted(set(
    glob.glob("/content/outputs/**/*.safetensors", recursive=True)
    + glob.glob("/content/sd-scripts/**/*.safetensors", recursive=True)
    + glob.glob("/content/**/sola_face*.safetensors", recursive=True)
))
# ignore huge base models if any
paths = [p for p in paths if os.path.getsize(p) < 500_000_000]
print("Found:")
for p in paths:
    print(round(os.path.getsize(p)/1e6, 2), "MB", p)

assert paths, "Still nothing — training did not save. Re-run cell 3 and read train.log"
best = [p for p in paths if "sola_face_sdxl" in os.path.basename(p)] or paths
best = sorted(best, key=lambda p: os.path.getmtime(p))[-1]
print("Downloading", best)
files.download(best)
